# Stage 07-B: Stage 1 — Contrastive Pretraining (ITC + Consistency)

**Purpose:** Train Vietnamese image-text alignment using COOLANT's contrastive learning objective
with PhoBERT as the text tower and COOLANT image features (64-dim) as the visual tower.
**No veracity labels are used** — this is self-supervised via in-batch negatives.

## Architecture

```
Statement text (NFC-normalised Vietnamese)
     |
PhoBERT-base-v2 (vinai/phobert-base-v2)
  layers 0–5 : FROZEN
  layers 6–11: trainable (lr = phobert_lr)
  CLS [B, 768]
     |
TextProjector: Linear[768 → 128] + LayerNorm + GeLU
  e_t [B, 128]  ← L2-normalised                              ImageProjector: Linear[64 → 128] + LayerNorm + GeLU
     |                                                                e_v [B, 128]  ← L2-normalised
     +──────────── ITC (bidirectional, τ=0.07) ─────────────────────+
                          L_CL  (Eq. 2-4 COOLANT paper)

Consistency (matched/unmatched binary classifier):
  cat(e_t, e_v, |e_t - e_v|) → Linear → L_ITM  (Eq. 1 COOLANT paper)
  In-batch negatives: shuffle image indices within batch
```

## Key decision rule (§7 research plan)

> If validation ITC top-1 accuracy (image→text retrieval within batch) **does not exceed ~30%**
> after 10 epochs (chance ≈ 3% for batch=32), stop and fall back to **Path A** rather than
> proceeding to Stage 2 — you'd be fine-tuning a classifier on a non-aligned embedding space.

## Prerequisites

- `07a` sanity check passed
- `06d_prepare_dataset.ipynb` built `stage06d_cache/{train,dev,test}.h5`

## Outputs

- `training/checkpoints_stage07b/best_pretrain.pt`  ← load in 07c
- `training/stage07b_results/pretrain_results.json`
- `training/stage07b_results/itc_accuracy_curve.png`

In [ ]:
# ─── Environment Setup (do not edit) ─────────────────────────────────────────
import os, sys
from pathlib import Path


def _detect_platform():
    try:
        import google.colab  # noqa: F401
        return 'colab', False
    except ImportError:
        pass
    if Path('/workspace').exists() and os.environ.get('VAST_CONTAINERLABEL'):
        return 'vastai', False
    if Path('/workspace').exists():
        return 'vastai', True
    if sys.platform == 'win32': return 'windows', False
    if sys.platform == 'darwin': return 'mac', False
    return None, True


PLATFORM, _uncertain = _detect_platform()
if PLATFORM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')

try:
    _nb_path = Path(__file__).resolve()
except NameError:
    _nb_path = Path.cwd()

PROJECT_ROOT = (
    Path('/content/drive/MyDrive/Thesis_Final/fake-news-detection') if PLATFORM == 'colab'
    else _nb_path.parents[1]
)
sys.path.insert(0, str(PROJECT_ROOT))

_env_map = {
    'colab':   PROJECT_ROOT / '.env.colab',
    'vastai':  PROJECT_ROOT / '.env.vastai',
    'windows': PROJECT_ROOT / '.env.windows',
    'mac':     PROJECT_ROOT / '.env.mac',
}
_env_file = _env_map.get(PLATFORM, PROJECT_ROOT / '.env')
if not _env_file.exists():
    _env_file = PROJECT_ROOT / '.env'

from dotenv import load_dotenv
load_dotenv(_env_file, override=True)
from src.utils.env_utils import get_data_root
DATA_ROOT = get_data_root()
print(f'Platform : {PLATFORM}  DATA_ROOT: {DATA_ROOT}')

In [ ]:
# ── Step 1: Configuration ─────────────────────────────────────────────────────
CONFIG = {
    'paths': {
        'stage06d_cache_dir': DATA_ROOT / 'training' / 'stage06d_cache',
        'checkpoint_dir':     DATA_ROOT / 'training' / 'checkpoints_stage07b',
        'results_dir':        DATA_ROOT / 'training' / 'stage07b_results',
        'mlflow_dir':         DATA_ROOT / 'mlruns',
    },
    'phobert': {
        'model_name':     'vinai/phobert-base-v2',
        'max_seq_len':    128,
        'freeze_n_layers': 6,    # freeze bottom 6 of 12 transformer layers
        'text_a_field':   'statement',
    },
    'model': {
        'image_dim':  64,     # COOLANT clip_embed_dim from stage06d_cache
        'text_dim':   768,    # PhoBERT-base hidden size
        'proj_dim':   128,    # shared projection dim (matches COOLANT)
        'temperature': 0.07,  # ITC temperature τ (COOLANT default)
        'itm_hidden': 256,    # consistency classifier hidden dim
        'dropout':    0.1,
    },
    'training': {
        'batch_size':    32,    # research plan: 32-64; 32 for balanced in-batch negatives
        'max_epochs':    50,    # Stage 1 can run longer (not label-bottlenecked)
        'patience':      8,     # early stopping on val ITC accuracy
        'phobert_lr':    2e-5,  # upper transformer layers
        'head_lr':       1e-4,  # projection heads
        'weight_decay':  0.01,  # AdamW
        'warmup_pct':    0.06,  # 6% of total steps
        'grad_clip':     1.0,
        'seed':          42,
        'lambda_itm':    0.5,   # weight for L_ITM (consistency loss)
    },
    'decision': {
        'itc_threshold': 0.30,  # min val ITC top-1 acc to proceed to Stage 2
        'check_epoch':   10,    # evaluate decision rule at this epoch
    },
    'mlflow': {'experiment_name': 'stage07b-contrastive-pretrain'},
    'safety': {'smoke_test': False, 'smoke_batches': 3, 'smoke_epochs': 2},
}
for d in ('checkpoint_dir', 'results_dir'):
    CONFIG['paths'][d].mkdir(parents=True, exist_ok=True)
print(
    f'freeze_n={CONFIG["phobert"]["freeze_n_layers"]}  '
    f'proj_dim={CONFIG["model"]["proj_dim"]}  '
    f'tau={CONFIG["model"]["temperature"]}  '
    f'lr: phobert={CONFIG["training"]["phobert_lr"]} / head={CONFIG["training"]["head_lr"]}'
)

In [ ]:
# ── Step 2: Dependencies, Device, Seed ────────────────────────────────────────
import importlib, unicodedata, random, json, gc
from datetime import datetime

_required = [
    ('torch', 'torch'), ('h5py', 'h5py'), ('numpy', 'numpy'),
    ('pandas', 'pandas'), ('matplotlib', 'matplotlib'),
    ('tqdm', 'tqdm'), ('transformers', 'transformers'),
]
_missing = [pkg for mod, pkg in _required if importlib.util.find_spec(mod) is None]
if _missing:
    raise RuntimeError(f'Missing packages: {_missing}')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


seed_everything(CONFIG['training']['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}  PyTorch: {torch.__version__}')

In [ ]:
# ── Step 3: Load Dataset (label-agnostic — Stage 1 is self-supervised) ────────
def load_pretrain_split(split, cfg):
    p = cfg['paths']['stage06d_cache_dir'] / f'{split}.h5'
    if not p.exists():
        raise FileNotFoundError(f'{p}\nRun 06d_prepare_dataset.ipynb first.')
    recs = []
    with h5py.File(p, 'r') as f:
        n = int(f.attrs['n_articles'])
        imgs  = f['image_features'][:]   # [N, max_pairs, 64]
        cnts  = f['pair_counts'][:]       # [N]
        stmts = [str(s) for s in f['statements'][:]]
    for i in range(n):
        pc  = int(cnts[i])
        vec = imgs[i][:pc].mean(axis=0).astype(np.float32) if pc > 0 else np.zeros(64, np.float32)
        recs.append({
            'text':           unicodedata.normalize('NFC', stmts[i]),
            'image':          vec,       # [64]
            'has_image':      pc > 0,
        })
    return recs


data = {s: load_pretrain_split(s, CONFIG) for s in ('train', 'dev')}
for split, recs in data.items():
    n_img = sum(1 for r in recs if r['has_image'])
    print(f'  [{split}] n={len(recs)}  with_image={n_img} ({100*n_img//len(recs)}%)')

In [ ]:
# ── Step 4: PhoBERT Tokenizer ─────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG['phobert']['model_name'], use_fast=True
)
print(f'Tokenizer: {CONFIG["phobert"]["model_name"]}  vocab={tokenizer.vocab_size}')

In [ ]:
# ── Step 5: Dataset + DataLoader ──────────────────────────────────────────────
class ContrastiveDataset(Dataset):
    def __init__(self, records, tokenizer, max_len):
        self.records   = records
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self): return len(self.records)

    def __getitem__(self, idx):
        r   = self.records[idx]
        enc = self.tokenizer(
            r['text'],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'image':          torch.tensor(r['image'], dtype=torch.float32),
            'has_image':      torch.tensor(float(r['has_image'])),
        }


max_len = CONFIG['phobert']['max_seq_len']
bs      = CONFIG['training']['batch_size']

datasets = {s: ContrastiveDataset(data[s], tokenizer, max_len) for s in ('train', 'dev')}
loaders  = {
    'train': DataLoader(datasets['train'], batch_size=bs, shuffle=True,  num_workers=2, pin_memory=True),
    'dev':   DataLoader(datasets['dev'],   batch_size=bs, shuffle=False, num_workers=2, pin_memory=True),
}
print(f'Train batches: {len(loaders["train"])}  Val batches: {len(loaders["dev"])}')

In [ ]:
# ── Step 6: ViCOOLANT Stage-1 Model (ITC + Consistency) ──────────────────────
class ViCOOLANTPretrain(nn.Module):
    """
    COOLANT §3.2 re-instantiated with PhoBERT as text tower.
    Implements:
      - Bidirectional ITC loss (Eq. 2-4 in COOLANT paper)
      - Consistency learning / ITM (Eq. 1) with in-batch negatives
    """
    def __init__(self, phobert_name, image_dim, text_dim, proj_dim, temperature,
                 itm_hidden, dropout, freeze_n_layers):
        super().__init__()
        self.temperature = temperature

        # Text encoder: PhoBERT
        self.text_encoder = AutoModel.from_pretrained(phobert_name)
        # Freeze bottom freeze_n_layers transformer layers
        for i, layer in enumerate(self.text_encoder.encoder.layer):
            if i < freeze_n_layers:
                for p in layer.parameters(): p.requires_grad = False
        # Always freeze embedding layer
        for p in self.text_encoder.embeddings.parameters(): p.requires_grad = False

        # Text projection head
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
        )

        # Image projection head
        self.image_proj = nn.Sequential(
            nn.Linear(image_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
        )

        # Consistency (ITM) classifier — operates on concat of both projections
        self.itm_head = nn.Sequential(
            nn.Linear(proj_dim * 3, itm_hidden),  # e_t, e_v, |e_t - e_v|
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(itm_hidden, 1),  # binary: matched=1, unmatched=0
        )

    def encode_text(self, input_ids, attention_mask):
        out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]   # [B, 768]
        return F.normalize(self.text_proj(cls), dim=-1)  # [B, proj_dim]

    def encode_image(self, image_feats):
        return F.normalize(self.image_proj(image_feats), dim=-1)  # [B, proj_dim]

    def itc_loss(self, e_t, e_v):
        """Bidirectional ITC (Eq. 2-4). Uses only image-present pairs."""
        logits = (e_v @ e_t.T) / self.temperature  # [B, B]
        labels = torch.arange(len(e_t), device=e_t.device)
        loss_i2t = F.cross_entropy(logits, labels)
        loss_t2i = F.cross_entropy(logits.T, labels)
        return (loss_i2t + loss_t2i) / 2

    def itm_loss(self, e_t, e_v):
        """Consistency learning (Eq. 1). Negatives = shuffled image indices in batch."""
        B = e_t.size(0)
        # Positive pairs: matched (label=1)
        pos_feat   = torch.cat([e_t, e_v, (e_t - e_v).abs()], dim=-1)  # [B, proj*3]
        pos_logits = self.itm_head(pos_feat).squeeze(-1)                # [B]
        pos_labels = torch.ones(B, device=e_t.device)
        # Negative pairs: shuffled image (label=0)
        shuffle_idx = torch.randperm(B, device=e_t.device)
        e_v_neg    = e_v[shuffle_idx]
        neg_feat   = torch.cat([e_t, e_v_neg, (e_t - e_v_neg).abs()], dim=-1)
        neg_logits = self.itm_head(neg_feat).squeeze(-1)                # [B]
        neg_labels = torch.zeros(B, device=e_t.device)
        return F.binary_cross_entropy_with_logits(
            torch.cat([pos_logits, neg_logits]),
            torch.cat([pos_labels, neg_labels]),
        )

    def forward(self, input_ids, attention_mask, image, has_image):
        e_t = self.encode_text(input_ids, attention_mask)  # [B, proj_dim]
        e_v = self.encode_image(image)                     # [B, proj_dim]

        # Use only has_image=True pairs for contrastive losses
        img_mask = has_image.bool()
        if img_mask.sum() < 2:
            return None, None  # batch too small — skip
        e_t_img = e_t[img_mask]
        e_v_img = e_v[img_mask]

        l_itc = self.itc_loss(e_t_img, e_v_img)
        l_itm = self.itm_loss(e_t_img, e_v_img)
        return l_itc, l_itm


model = ViCOOLANTPretrain(
    phobert_name=CONFIG['phobert']['model_name'],
    image_dim=CONFIG['model']['image_dim'],
    text_dim=CONFIG['model']['text_dim'],
    proj_dim=CONFIG['model']['proj_dim'],
    temperature=CONFIG['model']['temperature'],
    itm_hidden=CONFIG['model']['itm_hidden'],
    dropout=CONFIG['model']['dropout'],
    freeze_n_layers=CONFIG['phobert']['freeze_n_layers'],
).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Params: trainable={trainable:,}  total={total:,}  ({100*trainable//total}% unfrozen)')

In [ ]:
# ── Step 7: Optimizer + Scheduler (Differential LR) ─────────────────────────
phobert_params = [
    p for n, p in model.named_parameters()
    if p.requires_grad and 'text_encoder' in n
]
head_params = [
    p for n, p in model.named_parameters()
    if p.requires_grad and 'text_encoder' not in n
]

optimizer = torch.optim.AdamW([
    {'params': phobert_params, 'lr': CONFIG['training']['phobert_lr']},
    {'params': head_params,    'lr': CONFIG['training']['head_lr']},
], weight_decay=CONFIG['training']['weight_decay'])

total_steps  = len(loaders['train']) * CONFIG['training']['max_epochs']
warmup_steps = int(total_steps * CONFIG['training']['warmup_pct'])
scheduler    = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
print(f'Total steps={total_steps}  warmup={warmup_steps}')

In [ ]:
# ── Step 8: Validation — Top-1 ITC Accuracy ──────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, lambda_itm):
    model.eval()
    total_itc, total_itm, n_batches = 0.0, 0.0, 0
    top1_correct, top1_total = 0, 0

    for batch in loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        imgs = batch['image'].to(device)
        hi   = batch['has_image'].to(device)

        e_t  = model.encode_text(ids, mask)
        e_v  = model.encode_image(imgs)

        img_mask = hi.bool()
        if img_mask.sum() < 2:
            continue
        e_t_img = e_t[img_mask]
        e_v_img = e_v[img_mask]

        l_itc = model.itc_loss(e_t_img, e_v_img)
        l_itm = model.itm_loss(e_t_img, e_v_img)
        total_itc += l_itc.item()
        total_itm += l_itm.item()
        n_batches += 1

        # Top-1 image-to-text retrieval accuracy
        sim = e_v_img @ e_t_img.T                   # [M, M]
        pred = sim.argmax(dim=1)                     # [M]
        gt   = torch.arange(len(e_t_img), device=device)
        top1_correct += (pred == gt).sum().item()
        top1_total   += len(gt)

    if n_batches == 0:
        return {'val_itc': 0, 'val_itm': 0, 'val_total': 0, 'val_itc_top1': 0.0}

    avg_itc   = total_itc / n_batches
    avg_itm   = total_itm / n_batches
    itc_top1  = top1_correct / top1_total if top1_total else 0
    total_loss = avg_itc + lambda_itm * avg_itm
    return {
        'val_itc': avg_itc, 'val_itm': avg_itm,
        'val_total': total_loss, 'val_itc_top1': itc_top1,
    }


print('evaluate() defined')

In [ ]:
# ── Step 9: Training Loop ─────────────────────────────────────────────────────
lambda_itm  = CONFIG['training']['lambda_itm']
max_epochs  = CONFIG['training']['max_epochs']
patience    = CONFIG['training']['patience']
grad_clip   = CONFIG['training']['grad_clip']
smoke       = CONFIG['safety']['smoke_test']
smoke_b     = CONFIG['safety']['smoke_batches']
smoke_ep    = CONFIG['safety']['smoke_epochs']
ckpt_path   = CONFIG['paths']['checkpoint_dir'] / 'best_pretrain.pt'

history      = []
best_top1    = 0.0
no_improve   = 0
_max_epochs  = smoke_ep if smoke else max_epochs

for epoch in range(1, _max_epochs + 1):
    model.train()
    train_itc, train_itm, train_steps = 0.0, 0.0, 0

    pbar = tqdm(loaders['train'], desc=f'Epoch {epoch}/{_max_epochs}', leave=False)
    for step, batch in enumerate(pbar):
        if smoke and step >= smoke_b: break
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        imgs = batch['image'].to(device)
        hi   = batch['has_image'].to(device)

        optimizer.zero_grad()
        l_itc, l_itm = model(ids, mask, imgs, hi)
        if l_itc is None:
            continue  # batch had < 2 valid image pairs

        loss = l_itc + lambda_itm * l_itm
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        scheduler.step()

        train_itc   += l_itc.item()
        train_itm   += l_itm.item()
        train_steps += 1
        pbar.set_postfix(itc=f'{l_itc.item():.3f}', itm=f'{l_itm.item():.3f}')

    val_m = evaluate(model, loaders['dev'], lambda_itm)
    avg_itc = train_itc / max(train_steps, 1)
    avg_itm = train_itm / max(train_steps, 1)

    row = {
        'epoch':           epoch,
        'train_itc':       avg_itc,
        'train_itm':       avg_itm,
        'train_total':     avg_itc + lambda_itm * avg_itm,
        **{f'val_{k}': v for k, v in val_m.items()},
    }
    history.append(row)

    top1 = val_m['val_itc_top1']
    improved = top1 > best_top1
    if improved:
        best_top1  = top1
        no_improve = 0
        torch.save({
            'epoch':      epoch,
            'state_dict': model.state_dict(),
            'best_top1':  best_top1,
            'config':     CONFIG,
        }, ckpt_path)
    else:
        no_improve += 1

    print(
        f'Ep {epoch:3d} | train ITC={avg_itc:.4f} ITM={avg_itm:.4f} | '
        f'val ITC={val_m["val_itc"]:.4f} ITM={val_m["val_itm"]:.4f} | '
        f'ITC-top1={top1:.4f}{" *" if improved else ""}'
    )

    # Decision rule check at epoch 10
    if epoch == CONFIG['decision']['check_epoch'] and not smoke:
        thresh = CONFIG['decision']['itc_threshold']
        if best_top1 < thresh:
            print(f'[DECISION] Epoch {epoch}: best ITC top-1={best_top1:.4f} < {thresh}.')
            print('  Stopping — alignment signal too weak. Consider Path A fallback.')
            break

    if no_improve >= patience and not smoke:
        print(f'Early stopping at epoch {epoch} (patience={patience})')
        break

print(f'\nBest val ITC top-1: {best_top1:.4f}  checkpoint: {ckpt_path}')

In [ ]:
# ── Step 10: Training Curves ──────────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(df['epoch'], df['train_itc'], label='Train ITC', color='#3B82F6')
axes[0].plot(df['epoch'], df['val_itc'],   label='Val ITC',   color='#3B82F6', linestyle='--')
axes[0].plot(df['epoch'], df['train_itm'], label='Train ITM', color='#F59E0B')
axes[0].plot(df['epoch'], df['val_itm'],   label='Val ITM',   color='#F59E0B', linestyle='--')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Stage 1 — Training Losses'); axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

axes[1].plot(df['epoch'], df['val_itc_top1'], color='#10B981', linewidth=2, label='Val ITC Top-1')
axes[1].axhline(CONFIG['decision']['itc_threshold'], color='#EF4444', linestyle='--',
                linewidth=1.5, label=f'Threshold ({CONFIG["decision"]["itc_threshold"]})')
axes[1].axhline(1 / CONFIG['training']['batch_size'], color='gray', linestyle=':',
                linewidth=1, label=f'Chance (1/bs={1/CONFIG["training"]["batch_size"]:.3f})')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Top-1 Retrieval Accuracy')
axes[1].set_title('ITC Top-1 Image→Text Retrieval')
axes[1].set_ylim(0, min(1.05, df['val_itc_top1'].max() + 0.1))
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.suptitle('Stage 07-B: Contrastive Pretraining Curves', fontsize=12)
plt.tight_layout()
p = CONFIG['paths']['results_dir'] / 'itc_accuracy_curve.png'
plt.savefig(p, dpi=150); plt.show(); plt.close()
print(f'Saved: {p}')

In [ ]:
# ── Step 11: Save Results + Final Decision ────────────────────────────────────
thresh = CONFIG['decision']['itc_threshold']
results = {
    'generated':          datetime.now().isoformat(),
    'best_val_itc_top1':  best_top1,
    'epochs_trained':     len(history),
    'decision_threshold': thresh,
    'proceed_to_stage2':  best_top1 >= thresh,
    'checkpoint':         str(ckpt_path),
    'config':             {k: str(v) if isinstance(v, Path) else v for k, v in {
        'phobert_lr':    CONFIG['training']['phobert_lr'],
        'head_lr':       CONFIG['training']['head_lr'],
        'temperature':   CONFIG['model']['temperature'],
        'freeze_n':      CONFIG['phobert']['freeze_n_layers'],
        'proj_dim':      CONFIG['model']['proj_dim'],
        'batch_size':    CONFIG['training']['batch_size'],
    }.items()},
    'training_history':   history,
}

rp = CONFIG['paths']['results_dir'] / 'pretrain_results.json'
with open(rp, 'w') as f: json.dump(results, f, indent=2)

print('=' * 60)
print('STAGE 1 PRETRAINING — FINAL DECISION')
print('=' * 60)
print(f'Best val ITC top-1 : {best_top1:.4f}')
print(f'Threshold           : {thresh}')
if best_top1 >= thresh:
    print('[PASS] Proceed to 07c (Stage 2 supervised fine-tuning).')
    print(f'  Load checkpoint: {ckpt_path}')
else:
    print('[FAIL] ITC alignment below threshold.')
    print('  Recommended actions:')
    print('  1. Check image sourcing quality (are images truly related to claims?)')
    print('  2. Try increasing unlabeled image-text pairs for Stage 1')
    print('  3. Fall back to Path A: frozen COOLANT features + Vietnamese text model')
print('=' * 60)
print(f'Results saved: {rp}')